In [ ]:
import torch
import transformers
import pandas as pd

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Enable a GPU accelerator in Kaggle settings.")

In [ ]:
from pathlib import Path

for p in Path("/kaggle/input").rglob("train.csv"):
    if (p.parent / "val.csv").exists():
        print("Dataset directory:", p.parent)

In [ ]:
import pandas as pd
from pathlib import Path

# Use the actual path printed by Cell 2
DATA_DIR = Path("/kaggle/input/processed-dataset-tdl")

TRAIN_PATH = DATA_DIR / "train.csv"
VAL_PATH = DATA_DIR / "val.csv"

LABELS = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate",
]

# Load only training and validation data
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

print("Training shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("\nTraining columns:", train_df.columns.tolist())

# Validate required columns
required = {"comment_text", *LABELS}

assert required.issubset(train_df.columns), (
    f"Missing training columns: {required - set(train_df.columns)}"
)
assert required.issubset(val_df.columns), (
    f"Missing validation columns: {required - set(val_df.columns)}"
)

# Check missing comments
assert train_df["comment_text"].notna().all()
assert val_df["comment_text"].notna().all()

# Check labels contain only binary values
for label in LABELS:
    assert set(train_df[label].unique()).issubset({0, 1}), (
        f"Invalid training labels in {label}"
    )
    assert set(val_df[label].unique()).issubset({0, 1}), (
        f"Invalid validation labels in {label}"
    )

print("\nPositive labels in training:")
print(train_df[LABELS].sum())

print("\nPositive labels in validation:")
print(val_df[LABELS].sum())

print("\nDataset validation passed.")

In [ ]:
import torch
import pandas as pd
from pathlib import Path

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Training data:", TRAIN_PATH)
print("Validation data:", VAL_PATH)

print("Training rows:", len(train_df))
print("Validation rows:", len(val_df))

LABELS = [
    "toxic", "severe_toxic", "obscene",
    "threat", "insult", "identity_hate"
]

assert torch.cuda.is_available(), "Enable GPU in Kaggle settings."
assert "comment_text" in train_df.columns
assert all(label in train_df.columns for label in LABELS)

print("Ready for GPU fine-tuning.")

In [ ]:
%pip install -q transformers datasets accelerate scikit-learn

In [ ]:
import torch
import transformers
import datasets
import sklearn

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("CUDA:", torch.cuda.is_available())

In [ ]:
import os
import random
import re
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer

SEED = 42
BASE_MODEL = "distilbert-base-uncased"
LABELS = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate",
]

MAX_LENGTH = 128
TUNING_TRAIN_SIZE = 20_000

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

def preprocess_text(text):
    text = str(text).lower()
    return re.sub(r"\s+", " ", text).strip()

# Use the same fixed training subset for every trial
train_tune = (
    train_df.sample(n=min(TUNING_TRAIN_SIZE, len(train_df)),
                    random_state=SEED)
    .reset_index(drop=True)
)

val_tune = val_df.reset_index(drop=True).copy()

train_texts = train_tune["comment_text"].map(preprocess_text).tolist()
val_texts = val_tune["comment_text"].map(preprocess_text).tolist()

y_train = train_tune[LABELS].to_numpy(dtype=np.float32)
y_val = val_tune[LABELS].to_numpy(dtype=np.float32)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

print("Tuning training rows:", len(train_texts))
print("Validation rows:", len(val_texts))
print("Training labels shape:", y_train.shape)
print("Validation labels shape:", y_val.shape)
print("Label order:", LABELS)
print("Tokenizer:", BASE_MODEL)

assert y_train.shape[1] == 6
assert y_val.shape[1] == 6
assert np.isfinite(y_train).all() and np.isfinite(y_val).all()

print("Tuning data preparation passed.")

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH,
    return_tensors="pt",
)

val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH,
    return_tensors="pt",
)

class ToxicDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.labels[idx],
        }

train_dataset = ToxicDataset(train_encodings, y_train)
val_dataset = ToxicDataset(val_encodings, y_val)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("Tokenized shape:", train_encodings["input_ids"].shape)

In [ ]:
import gc
import json
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import f1_score, precision_score, recall_score

LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

OUTPUT_ROOT = Path("/kaggle/working/toxicity_tuning")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 8
EPOCHS = 2
WEIGHT_DECAY = 0.01
LEARNING_RATES = [1e-5, 2e-5, 3e-5]

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False
)

def evaluate_model(model):
    model.eval()
    all_probs, all_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            inputs = {
                "input_ids": batch["input_ids"].to("cuda"),
                "attention_mask": batch["attention_mask"].to("cuda"),
            }
            logits = model(**inputs).logits
            probs = torch.sigmoid(logits)

            all_probs.append(probs.cpu().numpy())
            all_labels.append(batch["labels"].numpy())

    probs = np.concatenate(all_probs)
    true_labels = np.concatenate(all_labels)
    pred_labels = (probs >= 0.5).astype(int)

    return {
        "macro_f1": float(f1_score(
            true_labels, pred_labels, average="macro", zero_division=0
        )),
        "micro_f1": float(f1_score(
            true_labels, pred_labels, average="micro", zero_division=0
        )),
        "macro_precision": float(precision_score(
            true_labels, pred_labels, average="macro", zero_division=0
        )),
        "macro_recall": float(recall_score(
            true_labels, pred_labels, average="macro", zero_division=0
        )),
    }

results = []

for lr in LEARNING_RATES:
    print(f"\nStarting experiment: learning_rate={lr}")

    model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=6,
        problem_type="multi_label_classification",
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    ).to("cuda")

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=WEIGHT_DECAY,
    )

    steps_per_epoch = len(train_loader)
    total_steps = steps_per_epoch * EPOCHS
    scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=1.0, end_factor=0.1,
        total_iters=max(total_steps, 1)
    )

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0

        for step, batch in enumerate(train_loader):
            optimizer.zero_grad(set_to_none=True)

            inputs = {
                "input_ids": batch["input_ids"].to("cuda"),
                "attention_mask": batch["attention_mask"].to("cuda"),
                "labels": batch["labels"].to("cuda"),
            }

            output = model(**inputs)
            loss = output.loss
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{EPOCHS}: train_loss={avg_train_loss:.4f}")

    metrics = evaluate_model(model)
    print("Validation metrics:", metrics)

    trial_dir = OUTPUT_ROOT / f"lr_{lr:.0e}".replace("-", "")
    trial_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(trial_dir)
    tokenizer.save_pretrained(trial_dir)

    results.append({
        "learning_rate": lr,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "weight_decay": WEIGHT_DECAY,
        **metrics,
    })

    del model, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()

results_df = pd.DataFrame(results).sort_values(
    "macro_f1", ascending=False
)
display(results_df)

results_df.to_csv(OUTPUT_ROOT / "tuning_results.csv", index=False)
print("Results saved to:", OUTPUT_ROOT / "tuning_results.csv")

In [ ]:
import pandas as pd
from pathlib import Path

# Your existing Kaggle dataset
DATA_DIR = Path("/kaggle/input/processed-dataset-tdl")

TRAIN_PATH = DATA_DIR / "train.csv"
VAL_PATH = DATA_DIR / "val.csv"

# Reload the datasets into the current notebook session
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

LABELS = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate",
]

print("Training rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Training columns:", train_df.columns.tolist())

assert "comment_text" in train_df.columns
assert "comment_text" in val_df.columns
assert all(label in train_df.columns for label in LABELS)
assert all(label in val_df.columns for label in LABELS)

print("Datasets loaded and validated successfully.")

In [ ]:

import gc
import json
import random
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, precision_score, recall_score

SEED = 42
LR = 2e-5
EPOCHS = 2
BATCH_SIZE = 8
WEIGHT_DECAY = 0.01
MAX_LENGTH = 128

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

LABELS = [
    "toxic", "severe_toxic", "obscene",
    "threat", "insult", "identity_hate"
]
ID2LABEL = {i: label for i, label in enumerate(LABELS)}
LABEL2ID = {label: i for i, label in enumerate(LABELS)}

# Use the entire training and validation datasets
full_train_texts = train_df["comment_text"].fillna("").astype(str).tolist()
full_val_texts = val_df["comment_text"].fillna("").astype(str).tolist()

y_train_full = train_df[LABELS].to_numpy(dtype=np.float32)
y_val_full = val_df[LABELS].to_numpy(dtype=np.float32)

# Keep this preprocessing consistent with the previous experiments.
# Before deployment, align it with your project's preprocessing.py.
import re

def preprocess_text(text):
    return re.sub(r"\s+", " ", str(text).lower()).strip()

full_train_texts = [preprocess_text(x) for x in full_train_texts]
full_val_texts = [preprocess_text(x) for x in full_val_texts]

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

train_enc = tokenizer(
    full_train_texts, truncation=True, padding="max_length",
    max_length=MAX_LENGTH, return_tensors="pt"
)
val_enc = tokenizer(
    full_val_texts, truncation=True, padding="max_length",
    max_length=MAX_LENGTH, return_tensors="pt"
)

class ToxicDataset(Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.enc["input_ids"][idx],
            "attention_mask": self.enc["attention_mask"][idx],
            "labels": self.labels[idx],
        }

train_dataset_full = ToxicDataset(train_enc, y_train_full)
val_dataset_full = ToxicDataset(val_enc, y_val_full)

train_loader_full = DataLoader(
    train_dataset_full, batch_size=BATCH_SIZE, shuffle=True
)
val_loader_full = DataLoader(
    val_dataset_full, batch_size=BATCH_SIZE, shuffle=False
)

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=6,
    problem_type="multi_label_classification",
    id2label=ID2LABEL,
    label2id=LABEL2ID,
).to("cuda")

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)

total_steps = len(train_loader_full) * EPOCHS
scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=1.0, end_factor=0.1,
    total_iters=max(total_steps, 1)
)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for step, batch in enumerate(train_loader_full):
        optimizer.zero_grad(set_to_none=True)

        inputs = {
            "input_ids": batch["input_ids"].to("cuda"),
            "attention_mask": batch["attention_mask"].to("cuda"),
            "labels": batch["labels"].to("cuda"),
        }

        outputs = model(**inputs)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()

        if (step + 1) % 1000 == 0:
            print(
                f"Epoch {epoch+1}/{EPOCHS}, "
                f"step {step+1}/{len(train_loader_full)}, "
                f"loss {loss.item():.4f}"
            )

    print(
        f"Epoch {epoch+1} complete; "
        f"average training loss: {running_loss / len(train_loader_full):.4f}"
    )

# Full validation evaluation
model.eval()
all_probs, all_targets = [], []

with torch.no_grad():
    for batch in val_loader_full:
        inputs = {
            "input_ids": batch["input_ids"].to("cuda"),
            "attention_mask": batch["attention_mask"].to("cuda"),
        }
        logits = model(**inputs).logits
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_targets.append(batch["labels"].numpy())

probs = np.concatenate(all_probs)
targets = np.concatenate(all_targets)
preds = (probs >= 0.5).astype(int)

metrics = {
    "macro_f1": float(f1_score(targets, preds, average="macro", zero_division=0)),
    "micro_f1": float(f1_score(targets, preds, average="micro", zero_division=0)),
    "macro_precision": float(precision_score(targets, preds, average="macro", zero_division=0)),
    "macro_recall": float(recall_score(targets, preds, average="macro", zero_division=0)),
    "per_label_f1": {
        label: float(f1_score(targets[:, i], preds[:, i], zero_division=0))
        for i, label in enumerate(LABELS)
    },
}

OUTPUT_DIR = Path("/kaggle/working/exp_v1_tuned")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

metadata = {
    "base_model": "distilbert-base-uncased",
    "learning_rate": LR,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "weight_decay": WEIGHT_DECAY,
    "max_length": MAX_LENGTH,
    "seed": SEED,
    "labels": LABELS,
    "threshold": 0.5,
    "training_rows": len(train_dataset_full),
    "validation_rows": len(val_dataset_full),
    "metrics": metrics,
}

with open(OUTPUT_DIR / "tuning_results.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("\nFull-data training and validation completed.")
print(json.dumps(metadata, indent=2))
print("Saved model to:", OUTPUT_DIR)


In [ ]:
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_DIR = Path("/kaggle/working/exp_v1_tuned")

assert (MODEL_DIR / "config.json").exists()
assert (MODEL_DIR / "model.safetensors").exists()

tokenizer_check = AutoTokenizer.from_pretrained(str(MODEL_DIR))
model_check = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_DIR)
)

print("Model loaded successfully.")
print("Labels:", model_check.config.id2label)
print("Number of labels:", model_check.config.num_labels)
print("Tokenizer loaded:", tokenizer_check is not None)

In [ ]:
import shutil
from pathlib import Path

MODEL_DIR = Path("/kaggle/working/exp_v1_tuned")
ZIP_PATH = "/kaggle/working/exp_v1_tuned.zip"

shutil.make_archive(
    ZIP_PATH.removesuffix(".zip"),
    "zip",
    root_dir=MODEL_DIR,
)

print("Archive created:", ZIP_PATH)
print("Archive size (MB):", round(Path(ZIP_PATH).stat().st_size / (1024 * 1024), 2))

In [ ]:
from IPython.display import FileLink, display
display(FileLink('/kaggle/working/exp_v1_tuned.zip'))

In [ ]:
from pathlib import Path
import shutil
import os

MODEL_DIR = Path("/kaggle/working/exp_v1_tuned")
ZIP_PATH = Path("/kaggle/working/exp_v1_tuned.zip")

assert (MODEL_DIR / "config.json").exists(), "Missing config.json"
assert (MODEL_DIR / "model.safetensors").exists(), "Missing model weights"

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

shutil.make_archive(
    str(ZIP_PATH.with_suffix("")),
    "zip",
    root_dir=str(MODEL_DIR),
)

print("ZIP exists:", ZIP_PATH.exists())
print("ZIP size (MB):", round(ZIP_PATH.stat().st_size / 1024**2, 2))
print("Files included:")
shutil.unpack_archive  # archive inspection is done in the next cell